#Read BRONZE LAYER

In [1]:
#path = "/lakehouse/default/Files/raw/customer360/"

customers_raw = spark.read.parquet("abfss://SanketM_IB_WS@onelake.dfs.fabric.microsoft.com/Ecommerce_lakehouse.Lakehouse/Files/Bronze/customers (1).parquet")
orders_raw = spark.read.parquet("abfss://SanketM_IB_WS@onelake.dfs.fabric.microsoft.com/Ecommerce_lakehouse.Lakehouse/Files/Bronze/orders.parquet")
payments_raw = spark.read.parquet("abfss://SanketM_IB_WS@onelake.dfs.fabric.microsoft.com/Ecommerce_lakehouse.Lakehouse/Files/Bronze/payments.parquet")
support_raw = spark.read.parquet("abfss://SanketM_IB_WS@onelake.dfs.fabric.microsoft.com/Ecommerce_lakehouse.Lakehouse/Files/Bronze/support_tickets.parquet")
web_raw = spark.read.parquet("abfss://SanketM_IB_WS@onelake.dfs.fabric.microsoft.com/Ecommerce_lakehouse.Lakehouse/Files/Bronze/web_activities.parquet")

StatementMeta(, 46451f7a-ec10-4c60-a1f3-b71a589e9b61, 3, Finished, Available, Finished, False)

# Customer_DF

In [3]:
display(customers_raw.limit(5))

StatementMeta(, 46451f7a-ec10-4c60-a1f3-b71a589e9b61, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1447c97a-c284-4ea9-adf6-b891e65d7ac9)

# orders_raw

In [4]:
display(customers_raw.limit(5))

StatementMeta(, 46451f7a-ec10-4c60-a1f3-b71a589e9b61, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1e4514e7-798f-4388-beb2-a605f1eae043)

#   payments_raw

In [5]:
display(payments_raw.limit(5))

StatementMeta(, 46451f7a-ec10-4c60-a1f3-b71a589e9b61, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b96b4e66-fc06-4792-b184-a16fe886f533)

# support_raw

In [6]:
display(support_raw
.limit(5))

StatementMeta(, 46451f7a-ec10-4c60-a1f3-b71a589e9b61, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, faa9e161-ed9d-46ae-8c35-7587b0218cd4)

# web_raw

In [7]:
display(web_raw
.limit(5))

StatementMeta(, 46451f7a-ec10-4c60-a1f3-b71a589e9b61, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f1b8bf44-0b42-4be1-ab4b-b6bc60340c61)

### Create Bronze Delta Tables

In [8]:

customers_raw.write.format("delta").mode("overwrite").saveAsTable("customers")
orders_raw.write.format("delta").mode("overwrite").saveAsTable("orders")
payments_raw.write.format("delta").mode("overwrite").saveAsTable("payments")
support_raw.write.format("delta").mode("overwrite").saveAsTable("support")
web_raw.write.format("delta").mode("overwrite").saveAsTable("web")

StatementMeta(, 46451f7a-ec10-4c60-a1f3-b71a589e9b61, 10, Finished, Available, Finished, False)

# Silver Layer – Clean & Normalize

#### customer data clean

In [9]:
display(customers_raw.limit(6))

StatementMeta(, 46451f7a-ec10-4c60-a1f3-b71a589e9b61, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b43f6eb4-bc31-4f0a-ac7d-771499277535)

In [13]:

from pyspark.sql.functions import *
from pyspark.sql.types import *



customers_clean = (
    customers_raw
    .withColumn("email", lower(trim(col("EMAIL"))))
    .withColumn("name", initcap(trim(col("name"))))
    .withColumn("gender", when(lower(col("gender")).isin("f", "female"), "Female")
                          .when(lower(col("gender")).isin("m", "male"), "Male")
                          .otherwise("Other"))
    .withColumn("dob", to_date(regexp_replace(col("dob"), "/", "-")))
    .withColumn("location", initcap(col("location")))
    .dropDuplicates(["customer_id"])
    .dropna(subset=["customer_id", "email"])
)
display(customers_clean.limit(6))

customers_clean.write.format("delta").mode("overwrite").saveAsTable("silver_customers")



StatementMeta(, 46451f7a-ec10-4c60-a1f3-b71a589e9b61, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c22a52dd-efef-4c61-913a-b19b9c0d0482)

### clean order table

In [14]:
%%sql
select * from orders limit 5

StatementMeta(, 46451f7a-ec10-4c60-a1f3-b71a589e9b61, 16, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 5 fields>

In [16]:
orders = spark.table("orders")
orders_clean = (
    orders
    .withColumn("order_date", 
                when(col("order_date").rlike("^\d{4}/\d{2}/\d{2}$"), to_date(col("order_date"), "yyyy/MM/dd"))
                .when(col("order_date").rlike("^\d{2}-\d{2}-\d{4}$"), to_date(col("order_date"), "dd-MM-yyyy"))
                .when(col("order_date").rlike("^\d{8}$"), to_date(col("order_date"), "yyyyMMdd"))
                .otherwise(to_date(col("order_date"), "yyyy-MM-dd")))
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .withColumn("status", initcap(col("status")))
    .dropna(subset=["customer_id", "order_date"])
    .dropDuplicates(["order_id"])
)

display(orders_clean.limit(5))

orders_clean.write.format("delta").mode("overwrite").saveAsTable("silver_orders")

StatementMeta(, 46451f7a-ec10-4c60-a1f3-b71a589e9b61, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cf7eaf2c-a836-4473-b922-055034f4385c)


###  Payments - clean

In [18]:
%%sql
select * from payments limit 5

StatementMeta(, 46451f7a-ec10-4c60-a1f3-b71a589e9b61, 20, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 6 fields>

In [20]:

payments = spark.table("payments")
payments_clean = (
    payments
    .withColumn("payment_date", to_date(regexp_replace(col("payment_date"), "/", "-")))
    .withColumn("payment_method", initcap(col("payment_method")))
    .replace({"creditcard": "Credit Card"}, subset=["payment_method"])
    .withColumn("payment_status", initcap(col("payment_status")))
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .dropna(subset=["customer_id", "payment_date", "amount"])
)
display(payments_clean.limit(5))

payments_clean.write.format("delta").mode("overwrite").saveAsTable("silver_payments")

StatementMeta(, 46451f7a-ec10-4c60-a1f3-b71a589e9b61, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 10c0ffe5-08e4-477d-a6bb-74f8131576c3)

### Clean support

In [21]:
%%sql
select * from Support

StatementMeta(, 46451f7a-ec10-4c60-a1f3-b71a589e9b61, 23, Finished, Available, Finished, False)

<Spark SQL result set with 15 rows and 5 fields>

In [23]:
support = spark.table("support")
support_clean = (
    support
    .withColumn("ticket_date", to_date(regexp_replace(col("ticket_date"), "/", "-")))
    .withColumn("issue_type", initcap(trim(col("issue_type"))))
    .withColumn("resolution_status", initcap(trim(col("resolution_status"))))
    .replace({"NA": None, "": None}, subset=["issue_type", "resolution_status"])
    .dropDuplicates(["ticket_id"])
    .dropna(subset=["customer_id", "ticket_date"])
)
display(support_clean.limit(5))
support_clean.write.format("delta").mode("overwrite").saveAsTable("silver_support")

StatementMeta(, 46451f7a-ec10-4c60-a1f3-b71a589e9b61, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 57d3fd5b-210e-4756-9779-cde2a12fe0f3)

## Clean - Web

In [25]:
%%sql
select * from web limit 5

StatementMeta(, 46451f7a-ec10-4c60-a1f3-b71a589e9b61, 27, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 5 fields>

In [27]:
web = spark.table("web")
web_clean = (
    web
    .withColumn("session_time", to_date(regexp_replace(col("session_time"), "/", "-")))
    .withColumn("page_viewed", lower(col("page_viewed")))
    .withColumn("device_type", initcap(col("device_type")))
    .dropDuplicates(["session_id"])
    .dropna(subset=["customer_id", "session_time", "page_viewed"])
)
display(web_clean.limit(5))
web_clean.write.format("delta").mode("overwrite").saveAsTable("silver_web")


StatementMeta(, 46451f7a-ec10-4c60-a1f3-b71a589e9b61, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6b6256e5-a8d5-4030-b977-9e3ad1dd9340)

## -----GOLD Table - AGGREGATE TABLE

In [32]:
from pyspark.sql.functions import col

# Read Silver tables
cust = spark.table("silver_customers").alias("c")
orders = spark.table("silver_orders").alias("o")
payments = spark.table("silver_payments").alias("p")
support = spark.table("silver_support").alias("s")
web = spark.table("silver_web").alias("w")

# Customer 360
customer360 = (
    cust
    .join(orders, col("c.customer_id") == col("o.customer_id"), "left")
    .join(payments, col("c.customer_id") == col("p.customer_id"), "left")
    .join(support, col("c.customer_id") == col("s.customer_id"), "left")
    .join(web, col("c.customer_id") == col("w.customer_id"), "left")
    
    .select(
        # Customer
        col("c.customer_id"),
        col("c.name"),
        col("c.email"),
        col("c.gender"),
        col("c.dob"),
        col("c.location"),

        # Orders
        col("o.order_id"),
        col("o.order_date"),
        col("o.amount").alias("order_amount"),
        col("o.status").alias("order_status"),

        # Payments
        col("p.payment_method"),
        col("p.payment_status"),
        col("p.amount").alias("payment_amount"),

        # Support
        col("s.ticket_id"),
        col("s.issue_type"),
        col("s.ticket_date"),
        col("s.resolution_status"),

        # Web
        col("w.page_viewed"),
        col("w.device_type"),
        col("w.session_time")
    )
)

display(customer360.limit(10))

customer360.write.format("delta").mode("overwrite").saveAsTable("gold_customer360")

StatementMeta(, 46451f7a-ec10-4c60-a1f3-b71a589e9b61, 34, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 09c1fe73-5002-4b4e-a18f-50dcce8123b5)